# 05a — Young vs. old classifier

Can a model tell old cells from young ones, within each cell type, for mice it has never seen?
One logistic regression per cell type; mice are held out together (`StratifiedGroupKFold`).
Features: the tissue's highly variable genes minus the six sex-chromosome genes (`ag.SEX_PATTERN`; removed by default since the critical review, because sex is tied to age group and *Xist* inverted some splits).
All functions live in `src/aging.py`. Runs on any tissue; the findings at the end are Lung's.

### 1. Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
sys.path.append("..")          # so "src" can be imported from the notebook folder

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from src import aging as ag    # shared functions: edit src/aging.py, not the notebooks

TISSUE = os.environ.get("TISSUE", "Lung")   # set by src/run_tissue.py; edit here to run by hand
P = ag.paths(TISSUE)                        # data/{TISSUE}/..., figure/{TISSUE}/...

adata = sc.read_h5ad(P.processed)

### 2. Cell types with enough cells and mice in both groups

In [ ]:
ok = ag.usable_cell_types(adata)
ok

### 3. Main report (with the tissue's reviewed exclusions; Lung: 30-M-2 out of myeloid types)

`auroc_mouse` is the headline; `acc_train` near 1.0 is expected and shows overfitting, not performance.

In [ ]:
report = pd.DataFrame({ct: ag.age_eval(adata, ct, **ag.main_kw(ct, TISSUE)) for ct in ok}).T
report = report.astype(float).round(3).sort_values("auroc_mouse", ascending=False)
report.to_csv(P.figure("age_report.csv"))
report

### 4. Sensitivity checks: excluded mice, sex, ambient genes

In [ ]:
clean = ag.feature_mask(adata, "clean")
males = adata.obs.loc[adata.obs["sex"] == "male", "mouse.id"].astype(str).unique().tolist()

res = pd.DataFrame(index=ok)
res["logreg"] = [ag.age_auc(adata, ct, **ag.main_kw(ct, TISSUE)) for ct in ok]
res["with_excluded_mice"] = [ag.age_auc(adata, ct) for ct in ok]
res["males_only"] = [ag.age_auc(adata, ct, keep_mice=males, **ag.main_kw(ct, TISSUE)) for ct in ok]
res["no_ambient_genes"] = [ag.age_auc(adata, ct, features=clean, **ag.main_kw(ct, TISSUE)) for ct in ok]
res.round(3)

### 5. Control: age labels shuffled between mice (should be ~0.5)

In [ ]:
res["shuffled"] = [np.nanmean([ag.age_auc(adata, ct, shuffle_seed=s, **ag.main_kw(ct, TISSUE)) for s in range(5)])
                   for ct in ok]
res.round(3)

### 6. Comparison model: LightGBM

In [ ]:
from lightgbm import LGBMClassifier

def lgbm():
    return LGBMClassifier(n_estimators=200, learning_rate=0.05, num_leaves=15,
                          subsample=0.8, subsample_freq=1, colsample_bytree=0.5, verbose=-1)

res["lgbm"] = [ag.age_auc(adata, ct, make_model=lgbm, **ag.main_kw(ct, TISSUE)) for ct in ok]
res.round(3)

### 7. Batch check: train on some age groups, test on unseen ones

Age and processing batch are tangled in TMS. A cell type only counts as having a real age signal if `cross_age` stays well above 0.5. Values far below 0.5 mean the model learned batch-specific differences that point the wrong way on new batches.

Two versions. `cross_age_all_splits` averages every split from `ag.cross_age_splits` (one young age plus some old ages in training, the rest in test) and is the same rule for every tissue, so the atlas compares tissues with it. `cross_age` uses the tissue's hand-picked splits if it has them (Lung: train 1m + 18m + 21m, test 3m + 30m, and the reverse), otherwise it equals `cross_age_all_splits`. NaN means not testable: the tissue lacks two young or two old ages.

In [ ]:
picked = ag.settings(TISSUE).get("cross_age_splits")    # None -> every generic split
res["cross_age"] = [ag.cross_age_auc(adata, ct, splits=picked, **ag.main_kw(ct, TISSUE)) for ct in ok]
res["cross_age_all_splits"] = [ag.cross_age_auc(adata, ct, **ag.main_kw(ct, TISSUE)) for ct in ok]
res["passes_batch_check"] = res["cross_age"] >= 0.7
res.round(3).sort_values("cross_age", ascending=False)

### 8. Save table and figure

In [ ]:
res.to_csv(P.figure("age_auroc.csv"))

order = res.sort_values("cross_age").index
ax = res.loc[order, ["logreg", "cross_age", "shuffled"]].plot.barh(figsize=(6, 6))
ax.axvline(0.5, color="gray", linestyle="--")
ax.set_xlabel("AUROC")
ax.set_title(f"{TISSUE}: mouse-held-out vs. unseen-age-group test")
ax.figure.savefig(P.figure("age_auroc.png"), dpi=200, bbox_inches="tight")

### Findings (Lung)

- **The batch check changes the picture.** On Lung's hand-picked splits, fibroblasts (0.98) and adventitial cells (0.95) pass clearly; type II pneumocytes (0.81) pass on only 125 cells; alveolar macrophages and bronchial smooth muscle (0.80 each) pass once the sex gene *Xist* is out.
- **Neutrophils pass (0.78) but are unreliable:** their shuffled control is 0.70, far above the expected 0.5.
- **Lymphocytes and monocytes do not pass.** B, T, NK cells, monocytes and dendritic cells score 0.31–0.53 on unseen age groups (all generic splits: 0.45–0.60); 6 of 14 types sit below 0.5. Their high mouse-held-out scores mostly reflect batch.
- **Per-mouse AUROC of 1.0 is not proof.** Several cell types reach it and then fail the batch check, because batch and age coincide.
- **Sex.** *Xist* is the only sex-chromosome gene among Lung's variable genes. With it in (the pre-review default), 10 of 14 types scored 0.07–0.19 lower on Lung's splits (B 0.15, alveolar macrophage 0.66), because the 21m mice are all female. Sex still hides some signal through autosomal genes: B and NK cells score higher in male mice only (0.77 and 0.89 vs 0.54 and 0.67).
- **Other checks:** removing ambient genes changes AUROC by at most 0.08 (B 0.54 → 0.62) and never rescues a cell type on the batch check; LightGBM is never better than logistic regression.